# 📚 ComicCraft — AI Comic Story Creator (Gemini)

This notebook builds the **ComicCraft** project structure from scratch in your Colab runtime and launches the Flask app with a public URL via `pyngrok`.

```
comiccraft/
├── app/
│   ├── main.py
│   ├── routes.py
│   ├── gemini_flash.py
│   ├── gemini_pro.py
│   ├── image_generator.py
│   ├── layout_builder.py
│   └── exporters.py
├── templates/
│   ├── index.html
│   ├── comic_preview.html
│   └── export_success.html
├── static/
│   ├── panels/      # Generated panel images
│   ├── exports/     # Exported comic PDFs
│   └── fonts/       # Fonts for PDF generation
├── .env             # Environment variables (API keys)
└── requirements.txt
```

**How the pipeline works:** a story idea → `gemini_pro.py` (Gemini Pro breaks it into a structured panel script) → `gemini_flash.py` (Gemini Flash writes tight image prompts & dialogue per panel, fast) → `image_generator.py` (Imagen renders each panel, with a placeholder fallback) → `layout_builder.py` (Pillow composites panels + speech bubbles into comic pages) → `exporters.py` (pages become a downloadable PDF).

**Before running:** get a free Gemini API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

## 1. Install dependencies

In [ ]:
!pip install -q flask python-dotenv google-generativeai pillow reportlab pyngrok


## 2. Create the project folder structure

In [ ]:
import os

PROJECT_ROOT = "/content/comiccraft"
for sub in ["app", "templates", "static/panels", "static/exports", "static/fonts"]:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())


## 3. Set your API keys
Run this cell and paste your key when prompted (it's hidden, not printed, and only lives in this runtime).

In [ ]:
import getpass

google_api_key = getpass.getpass("Enter your GOOGLE_API_KEY (Gemini): ")
ngrok_token = getpass.getpass("Enter your NGROK_AUTH_TOKEN (optional, press Enter to skip): ")

with open(".env", "w") as f:
    f.write(f"GOOGLE_API_KEY={google_api_key}\n")
    f.write(f"NGROK_AUTH_TOKEN={ngrok_token}\n")
    f.write("FLASK_SECRET_KEY=dev-secret-change-me\n")

print("Saved to .env")


## 4. Write `requirements.txt`

In [ ]:
%%writefile requirements.txt
flask==3.0.3
python-dotenv==1.0.1
google-generativeai==0.8.3
pillow==10.4.0
reportlab==4.2.5
pyngrok==7.2.0


## 5. Write the `app/` package

In [ ]:
%%writefile app/__init__.py



In [ ]:
%%writefile app/gemini_pro.py
"""
gemini_pro.py
--------------
Uses Gemini's "pro" model for the heavy-lifting creative task: turning a
loose story idea into a structured, panel-by-panel comic script.

Gemini Pro is used here (rather than Flash) because story structuring
benefits from stronger reasoning about pacing, character consistency,
and scene composition. Flash is reserved for lighter, faster jobs
(see gemini_flash.py).
"""

import json
import os
import re

import google.generativeai as genai

PRO_MODEL_NAME = os.environ.get("GEMINI_PRO_MODEL", "gemini-2.5-pro")

SCRIPT_SYSTEM_PROMPT = """You are a professional comic book writer and \
storyboard artist. Given a story idea, break it into a sequence of comic \
panels.

Respond with STRICT JSON only, no markdown fences, no commentary, matching \
this schema:

{
  "title": "string",
  "art_style": "string, a short visual style descriptor",
  "panels": [
    {
      "panel_number": 1,
      "scene_description": "detailed visual description for an image \
generator: setting, characters, action, camera angle, mood/lighting",
      "characters": ["list of character names present"],
      "dialogue": [
        {"speaker": "Character Name", "line": "spoken line"}
      ],
      "caption": "optional narration box text, or empty string"
    }
  ]
}

Rules:
- Produce exactly the number of panels requested.
- Keep scene_description vivid and specific enough to draw from, but under \
60 words.
- Keep dialogue short (comic-bubble length, under ~15 words per line).
- Maintain consistent character appearance/description across panels.
"""


def _configure():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. Add it to your .env file."
        )
    genai.configure(api_key=api_key)


def _extract_json(text: str) -> dict:
    """Gemini sometimes wraps JSON in ```json fences despite instructions;
    strip those defensively before parsing."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    return json.loads(cleaned)


def generate_comic_script(story_prompt: str, num_panels: int = 6,
                           art_style: str = "modern digital comic art"):
    """
    Calls Gemini Pro to convert a free-text story idea into a structured
    panel-by-panel script.

    Returns a dict matching SCRIPT_SYSTEM_PROMPT's schema.
    """
    _configure()
    model = genai.GenerativeModel(
        model_name=PRO_MODEL_NAME,
        system_instruction=SCRIPT_SYSTEM_PROMPT,
    )

    user_prompt = (
        f"Story idea: {story_prompt}\n"
        f"Number of panels required: {num_panels}\n"
        f"Preferred art style: {art_style}\n"
        "Return the JSON now."
    )

    response = model.generate_content(
        user_prompt,
        generation_config={
            "temperature": 0.9,
            "response_mime_type": "application/json",
        },
    )

    try:
        script = _extract_json(response.text)
    except (json.JSONDecodeError, AttributeError) as exc:
        raise RuntimeError(
            f"Gemini Pro did not return valid JSON: {exc}\nRaw: "
            f"{getattr(response, 'text', response)}"
        )

    script.setdefault("art_style", art_style)
    return script


In [ ]:
%%writefile app/gemini_flash.py
"""
gemini_flash.py
-----------------
Uses Gemini's "flash" model for small, latency-sensitive jobs that run
once per panel, where speed matters more than deep reasoning:

  1. Turning a panel's scene_description into a tight, image-generator-
     ready prompt (adding art-style/consistency keywords).
  2. Polishing/shortening dialogue so it fits a speech bubble.

Keeping these on Flash (instead of Pro) keeps per-panel latency low even
for comics with many panels.
"""

import os

import google.generativeai as genai

FLASH_MODEL_NAME = os.environ.get("GEMINI_FLASH_MODEL", "gemini-2.5-flash")


def _configure():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. Add it to your .env file."
        )
    genai.configure(api_key=api_key)


def _get_model():
    _configure()
    return genai.GenerativeModel(model_name=FLASH_MODEL_NAME)


def build_image_prompt(scene_description: str, art_style: str,
                        characters: list[str]) -> str:
    """
    Expands a short scene description into a self-contained prompt for
    the image generator, folding in art style and character list so each
    panel stays visually consistent.
    """
    model = _get_model()
    cast = ", ".join(characters) if characters else "no named characters"
    prompt = (
        "Rewrite the following comic panel description as a single, dense "
        "image-generation prompt (one paragraph, no preamble, no quotes). "
        f"Art style to enforce: {art_style}. Characters present: {cast}.\n\n"
        f"Scene: {scene_description}"
    )
    response = model.generate_content(
        prompt, generation_config={"temperature": 0.6}
    )
    return response.text.strip()


def tighten_dialogue(line: str, max_words: int = 15) -> str:
    """Shortens a dialogue line to fit a speech bubble, if needed."""
    if len(line.split()) <= max_words:
        return line
    model = _get_model()
    prompt = (
        f"Shorten this comic dialogue line to at most {max_words} words, "
        "keeping the meaning and voice. Reply with only the shortened "
        f"line, no quotes:\n\n{line}"
    )
    response = model.generate_content(
        prompt, generation_config={"temperature": 0.3}
    )
    return response.text.strip()


In [ ]:
%%writefile app/image_generator.py
"""
image_generator.py
--------------------
Generates the artwork for each comic panel.

Primary path: Google's Imagen model via google-generativeai.
Fallback path: if image generation is unavailable (no access on your API
key/tier, quota, or network error), a clearly-labeled placeholder panel
is drawn with Pillow instead, so the pipeline never hard-fails end to
end. Swap IMAGE_MODEL_NAME or replace generate_panel_image's body with
any other image API you have access to.
"""

import os
import textwrap
import uuid

from PIL import Image, ImageDraw, ImageFont

import google.generativeai as genai

IMAGE_MODEL_NAME = os.environ.get("GEMINI_IMAGE_MODEL", "imagen-3.0-generate-002")
PANEL_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
    "static", "panels",
)
PANEL_SIZE = (768, 768)


def _configure():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. Add it to your .env file."
        )
    genai.configure(api_key=api_key)


def _placeholder_panel(prompt: str, panel_number: int) -> Image.Image:
    """Draws a simple placeholder panel so the app keeps working even
    without image-generation access. Not meant to look good -- just to
    keep the layout/export pipeline testable end to end."""
    img = Image.new("RGB", PANEL_SIZE, color=(235, 235, 235))
    draw = ImageDraw.Draw(img)
    draw.rectangle(
        [8, 8, PANEL_SIZE[0] - 8, PANEL_SIZE[1] - 8],
        outline=(20, 20, 20), width=6,
    )
    try:
        font = ImageFont.truetype(
            os.path.join(os.path.dirname(PANEL_DIR), "fonts", "Comic.ttf"),
            22,
        )
    except OSError:
        font = ImageFont.load_default()

    draw.text((24, 24), f"PANEL {panel_number} (placeholder)",
               fill=(180, 0, 0), font=font)
    wrapped = textwrap.fill(prompt, width=48)
    draw.text((24, 64), wrapped, fill=(30, 30, 30), font=font)
    return img


def generate_panel_image(prompt: str, panel_number: int) -> str:
    """
    Generates a single panel image and saves it under static/panels/.
    Returns the saved file path.
    """
    os.makedirs(PANEL_DIR, exist_ok=True)
    filename = f"panel_{panel_number}_{uuid.uuid4().hex[:8]}.png"
    out_path = os.path.join(PANEL_DIR, filename)

    try:
        _configure()
        result = genai.ImageGenerationModel(IMAGE_MODEL_NAME).generate_images(
            prompt=prompt,
            number_of_images=1,
            aspect_ratio="1:1",
        )
        image_bytes = result.images[0]._image_bytes  # library-internal but stable
        with open(out_path, "wb") as f:
            f.write(image_bytes)
    except Exception as exc:  # noqa: BLE001 - deliberately broad, see fallback
        print(f"[image_generator] Falling back to placeholder for panel "
              f"{panel_number}: {exc}")
        img = _placeholder_panel(prompt, panel_number)
        img.save(out_path)

    return out_path


def generate_all_panels(script: dict, prompts: list[str]) -> list[str]:
    """Generates an image for every panel in the script. `prompts` must be
    aligned with script['panels'] (same order/length)."""
    paths = []
    for panel, prompt in zip(script["panels"], prompts):
        paths.append(generate_panel_image(prompt, panel["panel_number"]))
    return paths


In [ ]:
%%writefile app/layout_builder.py
"""
layout_builder.py
--------------------
Takes the generated panel images plus their dialogue/captions and lays
them out into finished comic page images (a grid of panels per page,
with speech bubbles and caption boxes drawn on top).
"""

import os
import textwrap

from PIL import Image, ImageDraw, ImageFont

STATIC_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "static"
)
FONT_PATH = os.path.join(STATIC_DIR, "fonts", "Comic.ttf")

PANELS_PER_ROW = 2
PANEL_MARGIN = 20
GUTTER = 16
CAPTION_HEIGHT = 40
BUBBLE_PADDING = 10


def _load_font(size: int) -> ImageFont.FreeTypeFont:
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except OSError:
        return ImageFont.load_default()


def _draw_caption(draw: ImageDraw.ImageDraw, box, text, font):
    x0, y0, x1, y1 = box
    draw.rectangle(box, fill=(255, 233, 150), outline=(0, 0, 0), width=2)
    wrapped = textwrap.fill(text, width=42)
    draw.multiline_text((x0 + 8, y0 + 6), wrapped, fill=(0, 0, 0), font=font)


def _draw_speech_bubble(draw: ImageDraw.ImageDraw, anchor_xy, text, font,
                         panel_width: int):
    wrapped = textwrap.fill(text, width=28)
    lines = wrapped.split("\n")
    line_height = font.size + 4
    text_w = max(draw.textlength(line, font=font) for line in lines)
    text_h = line_height * len(lines)

    x, y = anchor_xy
    box = (
        x, y,
        min(x + text_w + BUBBLE_PADDING * 2, panel_width - 4),
        y + text_h + BUBBLE_PADDING * 2,
    )
    draw.rounded_rectangle(box, radius=14, fill=(255, 255, 255),
                            outline=(0, 0, 0), width=2)
    draw.multiline_text((box[0] + BUBBLE_PADDING, box[1] + BUBBLE_PADDING),
                         wrapped, fill=(0, 0, 0), font=font)
    return box[3] + 6  # bottom y, for stacking multiple bubbles


def compose_panel_with_text(panel_image_path: str, panel: dict) -> Image.Image:
    """Overlays captions and dialogue bubbles onto one panel image."""
    img = Image.open(panel_image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    font = _load_font(18)
    caption_font = _load_font(16)

    caption = panel.get("caption") or ""
    if caption:
        _draw_caption(draw, (0, 0, img.width, CAPTION_HEIGHT), caption,
                      caption_font)

    y_cursor = img.height - 90
    dialogue = panel.get("dialogue") or []
    # stack bubbles from the bottom up
    y_cursor = img.height - (len(dialogue) * 70) - 10
    for line in dialogue:
        text = f"{line.get('speaker', '')}: {line.get('line', '')}".strip(": ")
        y_cursor = _draw_speech_bubble(draw, (10, max(y_cursor, CAPTION_HEIGHT + 10)),
                                        text, font, img.width)

    return img


def build_comic_pages(script: dict, panel_image_paths: list[str],
                       output_prefix: str) -> list[str]:
    """
    Arranges all panels (with text overlaid) into one or more comic page
    images, PANELS_PER_ROW per row. Saves pages under static/panels/ and
    returns their file paths in reading order.
    """
    panels = script["panels"]
    composed = [
        compose_panel_with_text(path, panel)
        for path, panel in zip(panel_image_paths, panels)
    ]

    panel_w, panel_h = composed[0].size if composed else (768, 768)
    panels_per_page = PANELS_PER_ROW * 3  # 2x3 grid per page
    page_paths = []

    for page_idx in range(0, len(composed), panels_per_page):
        page_panels = composed[page_idx: page_idx + panels_per_page]
        rows = (len(page_panels) + PANELS_PER_ROW - 1) // PANELS_PER_ROW

        page_w = PANELS_PER_ROW * panel_w + (PANELS_PER_ROW + 1) * GUTTER
        page_h = rows * panel_h + (rows + 1) * GUTTER + PANEL_MARGIN * 2

        page = Image.new("RGB", (page_w, page_h + PANEL_MARGIN), "white")

        for i, panel_img in enumerate(page_panels):
            row, col = divmod(i, PANELS_PER_ROW)
            x = GUTTER + col * (panel_w + GUTTER)
            y = PANEL_MARGIN + GUTTER + row * (panel_h + GUTTER)
            page.paste(panel_img, (x, y))

        out_path = os.path.join(
            STATIC_DIR, "panels", f"{output_prefix}_page{page_idx // panels_per_page + 1}.png"
        )
        page.save(out_path)
        page_paths.append(out_path)

    return page_paths


In [ ]:
%%writefile app/exporters.py
"""
exporters.py
--------------
Exports the finished comic (a sequence of page images) to a single
downloadable PDF file under static/exports/.
"""

import os
import uuid

from PIL import Image
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

EXPORT_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
    "static", "exports",
)


def export_comic_to_pdf(page_image_paths: list[str], title: str = "comic") -> str:
    """
    Combines page images into one PDF, one comic page per PDF page,
    scaled to fit while preserving aspect ratio. Returns the saved
    PDF's file path.
    """
    os.makedirs(EXPORT_DIR, exist_ok=True)
    safe_title = "".join(c for c in title if c.isalnum() or c in " _-").strip() or "comic"
    filename = f"{safe_title.replace(' ', '_')}_{uuid.uuid4().hex[:8]}.pdf"
    out_path = os.path.join(EXPORT_DIR, filename)

    c = canvas.Canvas(out_path, pagesize=letter)
    page_w, page_h = letter
    margin = 36

    for path in page_image_paths:
        img = Image.open(path)
        img_w, img_h = img.size

        max_w, max_h = page_w - 2 * margin, page_h - 2 * margin
        scale = min(max_w / img_w, max_h / img_h)
        draw_w, draw_h = img_w * scale, img_h * scale
        x = (page_w - draw_w) / 2
        y = (page_h - draw_h) / 2

        c.drawImage(ImageReader(img), x, y, width=draw_w, height=draw_h)
        c.showPage()

    c.save()
    return out_path


In [ ]:
%%writefile app/routes.py
"""
routes.py
-----------
Flask routes wiring the full ComicCraft pipeline together:

  index (GET)          -> story input form
  generate (POST)       -> gemini_pro -> gemini_flash -> image_generator
                            -> layout_builder, then show comic_preview
  export/<comic_id> (POST) -> exporters -> export_success

Generated comics are kept in-memory (COMICS dict) for this simple demo;
for real usage you'd persist them (DB, session, etc.).
"""

import os
import uuid

from flask import Blueprint, render_template, request, redirect, url_for, flash

from . import gemini_pro, gemini_flash, image_generator, layout_builder, exporters

bp = Blueprint("comiccraft", __name__)

# In-memory store: comic_id -> {"script": ..., "page_paths": [...]}
COMICS = {}


@bp.route("/", methods=["GET"])
def index():
    return render_template("index.html")


@bp.route("/generate", methods=["POST"])
def generate():
    story_prompt = request.form.get("story_prompt", "").strip()
    num_panels = int(request.form.get("num_panels", 6))
    art_style = request.form.get("art_style", "modern digital comic art").strip()

    if not story_prompt:
        flash("Please enter a story idea.")
        return redirect(url_for("comiccraft.index"))

    # 1. Gemini Pro: story -> structured panel script
    script = gemini_pro.generate_comic_script(story_prompt, num_panels, art_style)

    # 2. Gemini Flash: per-panel image prompt + dialogue tightening
    image_prompts = []
    for panel in script["panels"]:
        prompt = gemini_flash.build_image_prompt(
            panel["scene_description"], script.get("art_style", art_style),
            panel.get("characters", []),
        )
        image_prompts.append(prompt)
        panel["dialogue"] = [
            {"speaker": d.get("speaker", ""),
             "line": gemini_flash.tighten_dialogue(d.get("line", ""))}
            for d in panel.get("dialogue", [])
        ]

    # 3. Image generation per panel
    panel_image_paths = image_generator.generate_all_panels(script, image_prompts)

    # 4. Layout: compose panels (+text) into comic page(s)
    comic_id = uuid.uuid4().hex[:10]
    page_paths = layout_builder.build_comic_pages(
        script, panel_image_paths, output_prefix=comic_id
    )

    COMICS[comic_id] = {"script": script, "page_paths": page_paths}

    page_urls = [
        url_for("static", filename=os.path.relpath(p, start=os.path.join(
            os.path.dirname(os.path.dirname(__file__)), "static")))
        for p in page_paths
    ]

    return render_template(
        "comic_preview.html",
        comic_id=comic_id,
        title=script.get("title", "Untitled Comic"),
        page_urls=page_urls,
    )


@bp.route("/export/<comic_id>", methods=["POST"])
def export(comic_id):
    comic = COMICS.get(comic_id)
    if not comic:
        flash("Comic not found -- generate one first.")
        return redirect(url_for("comiccraft.index"))

    pdf_path = exporters.export_comic_to_pdf(
        comic["page_paths"], title=comic["script"].get("title", "comic")
    )
    pdf_url = url_for("static", filename=os.path.relpath(
        pdf_path, start=os.path.join(os.path.dirname(os.path.dirname(__file__)), "static")
    ))

    return render_template("export_success.html", pdf_url=pdf_url)


In [ ]:
%%writefile app/main.py
"""
main.py
---------
ComicCraft Flask app factory / entry point.

Local / Colab usage:
    python -m app.main
(or see the Colab notebook, which also wires up pyngrok for a public URL)
"""

import os

from dotenv import load_dotenv
from flask import Flask

load_dotenv()  # loads GOOGLE_API_KEY etc. from .env


def create_app() -> Flask:
    base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    app = Flask(
        __name__,
        template_folder=os.path.join(base_dir, "templates"),
        static_folder=os.path.join(base_dir, "static"),
    )
    app.secret_key = os.environ.get("FLASK_SECRET_KEY", "dev-secret-change-me")

    from .routes import bp as comiccraft_bp
    app.register_blueprint(comiccraft_bp)

    return app


app = create_app()

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 5000)), debug=True)


## 6. Write the `templates/`

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>ComicCraft — AI Comic Story Creator</title>
  <style>
    body { font-family: 'Segoe UI', sans-serif; background: #1a1a2e; color: #eee;
           display: flex; justify-content: center; padding: 40px 16px; }
    .card { background: #16213e; padding: 32px; border-radius: 12px; max-width: 560px; width: 100%; }
    h1 { color: #ffca3a; margin-top: 0; }
    label { display: block; margin: 16px 0 6px; font-weight: 600; }
    input[type=text], textarea, select {
      width: 100%; padding: 10px; border-radius: 6px; border: none; box-sizing: border-box;
      font-size: 15px;
    }
    textarea { min-height: 100px; resize: vertical; }
    button {
      margin-top: 24px; padding: 12px 24px; background: #ff6b6b; color: white;
      border: none; border-radius: 6px; font-size: 16px; cursor: pointer; font-weight: 700;
    }
    button:hover { background: #ff8787; }
    .flash { background: #533; padding: 10px; border-radius: 6px; margin-bottom: 16px; }
  </style>
</head>
<body>
  <div class="card">
    <h1>📚 ComicCraft</h1>
    <p>Turn a story idea into an illustrated comic, powered by Gemini.</p>

    {% with messages = get_flashed_messages() %}
      {% if messages %}
        {% for m in messages %}<div class="flash">{{ m }}</div>{% endfor %}
      {% endif %}
    {% endwith %}

    <form action="{{ url_for('comiccraft.generate') }}" method="post">
      <label for="story_prompt">Story idea</label>
      <textarea id="story_prompt" name="story_prompt" placeholder="A shy robot discovers a hidden garden on an abandoned space station..." required></textarea>

      <label for="num_panels">Number of panels</label>
      <select id="num_panels" name="num_panels">
        <option value="4">4</option>
        <option value="6" selected>6</option>
        <option value="9">9</option>
        <option value="12">12</option>
      </select>

      <label for="art_style">Art style</label>
      <input type="text" id="art_style" name="art_style" value="modern digital comic art, bold inks, flat colors">

      <button type="submit">Generate Comic ✨</button>
    </form>
  </div>
</body>
</html>


In [ ]:
%%writefile templates/comic_preview.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>{{ title }} — ComicCraft Preview</title>
  <style>
    body { font-family: 'Segoe UI', sans-serif; background: #1a1a2e; color: #eee;
           display: flex; flex-direction: column; align-items: center; padding: 40px 16px; }
    h1 { color: #ffca3a; }
    img { max-width: 100%; margin: 16px 0; border-radius: 8px; box-shadow: 0 4px 14px rgba(0,0,0,.4); }
    .pages { max-width: 820px; width: 100%; }
    form { margin-top: 24px; }
    button, a.button {
      padding: 12px 24px; background: #06d6a0; color: #0a0a0a; border: none; border-radius: 6px;
      font-size: 16px; cursor: pointer; font-weight: 700; text-decoration: none; display: inline-block;
    }
    a.back { color: #aaa; margin-left: 16px; }
  </style>
</head>
<body>
  <h1>{{ title }}</h1>
  <div class="pages">
    {% for url in page_urls %}
      <img src="{{ url }}" alt="Comic page">
    {% endfor %}
  </div>

  <form action="{{ url_for('comiccraft.export', comic_id=comic_id) }}" method="post">
    <button type="submit">Export as PDF 📄</button>
    <a class="back" href="{{ url_for('comiccraft.index') }}">&larr; Start a new comic</a>
  </form>
</body>
</html>


In [ ]:
%%writefile templates/export_success.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Export Complete — ComicCraft</title>
  <style>
    body { font-family: 'Segoe UI', sans-serif; background: #1a1a2e; color: #eee;
           display: flex; flex-direction: column; align-items: center; justify-content: center;
           height: 100vh; text-align: center; }
    h1 { color: #06d6a0; }
    a.button {
      margin-top: 20px; padding: 12px 24px; background: #ffca3a; color: #0a0a0a;
      border-radius: 6px; font-size: 16px; font-weight: 700; text-decoration: none;
    }
    a.back { display: block; margin-top: 16px; color: #aaa; }
  </style>
</head>
<body>
  <h1>✅ Your comic is ready!</h1>
  <a class="button" href="{{ pdf_url }}" download>Download PDF</a>
  <a class="back" href="{{ url_for('comiccraft.index') }}">&larr; Create another comic</a>
</body>
</html>


## 7. (Optional) Add a comic-style font
Drop any free TTF (e.g. a comic-lettering font) into `static/fonts/Comic.ttf` for nicer speech bubbles. If you skip this, everything still works with a default font.

In [ ]:
print("Skipping custom font -- Pillow's default font will be used.")
print("To add one: upload a .ttf via the Colab file browser to", 
      os.path.join(PROJECT_ROOT, 'static/fonts/Comic.ttf'))


## 8. Launch the Flask app with a public URL
This starts ComicCraft in the background and opens an `ngrok` tunnel so you can use the web UI from your browser.

In [ ]:
import os
import threading
from pyngrok import ngrok, conf

os.environ["GOOGLE_API_KEY"] = google_api_key
if ngrok_token:
    conf.get_default().auth_token = ngrok_token

import sys
sys.path.insert(0, PROJECT_ROOT)
from app.main import create_app

flask_app = create_app()

def run_app():
    flask_app.run(host="0.0.0.0", port=5000, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()

public_url = ngrok.connect(5000)
print("ComicCraft is live at:", public_url)


## 9. (Optional) Generate a comic directly in the notebook
Prefer code over the web UI? Call the pipeline modules directly:

In [ ]:
from app import gemini_pro, gemini_flash, image_generator, layout_builder, exporters

story_prompt = "A shy robot discovers a hidden garden on an abandoned space station"
num_panels = 4
art_style = "modern digital comic art, bold inks, flat colors"

script = gemini_pro.generate_comic_script(story_prompt, num_panels, art_style)

image_prompts = []
for panel in script["panels"]:
    prompt = gemini_flash.build_image_prompt(
        panel["scene_description"], script.get("art_style", art_style), panel.get("characters", [])
    )
    image_prompts.append(prompt)

panel_paths = image_generator.generate_all_panels(script, image_prompts)
page_paths = layout_builder.build_comic_pages(script, panel_paths, output_prefix="notebook_demo")
pdf_path = exporters.export_comic_to_pdf(page_paths, title=script.get("title", "comic"))

print("PDF saved to:", pdf_path)

from IPython.display import Image as IPImage, display
for p in page_paths:
    display(IPImage(filename=p))


## Notes
- **Image generation** uses Imagen via `google-generativeai`. If your API key/tier doesn't have image-gen access, `image_generator.py` automatically falls back to a labeled placeholder panel so the rest of the pipeline (layout + PDF export) still works end to end — swap in any other image API there if you prefer.
- **Models used:** `gemini-2.5-pro` for story structuring, `gemini-2.5-flash` for per-panel prompt/dialogue polishing (override via the `GEMINI_PRO_MODEL` / `GEMINI_FLASH_MODEL` / `GEMINI_IMAGE_MODEL` env vars if you want different ones).
- The Flask app keeps generated comics in memory (`COMICS` dict in `routes.py`) — fine for a demo/Colab session, but add a database if you need persistence.
- Re-run cell 8 if the ngrok tunnel drops.